In [ ]:
'''
AA-CBR evaluation using a trained 2D slot attention model using OS scores as outcomes.

Sweeps all combinations of char model, n_bins, agg_mode, strategy, and strict
but derives features from a trained SlotClassifier2D
instead of ground-truth segmentation masks.

Set CHECKPOINT below to your .pt file before running.
'''

import csv as csv_mod
import re
import sys
from itertools import product as iproduct
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

FYP_ROOT = Path('/path/to/BrainWear_Kareem/FYP')
if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))

from aacbr.aacbr_parallel import AACBRParallel
from aacbr.configs.brats_model_config import BraTSOutcomeConfig
from datasets.brats2020_png import BraTS2020PNGDataset
from utils.characterisations import (
    TumourCharacterisationLarge2D,
    TumourCharacterisationSmall2D,
    TumourCharacterisationSmall2DV2,
    TumourCharacterisationLarge2DV2,
)

CHAR_MODELS = {
    'small':    TumourCharacterisationSmall2D,
    'large':    TumourCharacterisationLarge2D,
    'small_v2': TumourCharacterisationSmall2DV2,
    'large_v2': TumourCharacterisationLarge2DV2,
}

DATA_DIR = str(
    Path('/path/to/BrainWear_Kareem')
    / 'Processed_BraTS2020_TrainingData_PNG'
)
OS_CSV = str(
    Path('/path/to/BrainWear_Kareem') / 'BraTS_OS.csv'
)
CONFIG = str(FYP_ROOT / 'aacbr' / 'configs' / 'brats_configs' / 'flat_config.json')
DEFAULT_TRAINED_LEADERBOARD = str(FYP_ROOT / 'aacbr' / 'leaderboard_trained.json')

In [ ]:
def load_slot_model(checkpoint_path: str, device: torch.device):
    from slot_attention.training_2d.slot_attention_2d import SlotClassifier2D

    ckpt = torch.load(checkpoint_path, map_location=device)
    hp = ckpt.get('hyperparameters', {})
    model = SlotClassifier2D(
        in_shape=hp.get('in_shape', (1, 240, 240)),
        width=hp.get('width', 64),
        num_slots=hp.get('num_slots', 5),
        slot_dim=hp.get('slot_dim', 64),
        routing_iters=hp.get('routing_iters', 7),
        temperature=hp.get('temperature', 0.5),
        encoder_depth=hp.get('encoder_depth', 4),
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device).eval()
    epoch = ckpt.get('epoch', '?')
    print(f'Loaded 2D slot model from {checkpoint_path}  (epoch {epoch})')
    return model


def _parse_survival_days(raw: str) -> float | None:
    raw = raw.strip()
    try:
        return float(raw)
    except ValueError:
        m = re.search(r'(\d+)', raw)
        return float(m.group(1)) if m else None


def load_os_csv(
    csv_path: str,
    n_bins: int = 5,
    resection_filter: str | None = None,
) -> tuple[dict[str, int], np.ndarray]:
    rows = []
    with open(csv_path, newline='', encoding='utf-8') as f:
        for row in csv_mod.DictReader(f):
            pid = row['Brats20ID'].strip()
            days = _parse_survival_days(row['Survival_days'])
            resection = row['Extent_of_Resection'].strip()
            if days is None:
                continue
            if resection_filter and resection != resection_filter:
                continue
            rows.append((pid, days))

    if not rows:
        raise ValueError('No usable rows found in OS CSV after filtering.')

    pids = [r[0] for r in rows]
    survival = np.array([r[1] for r in rows])
    quantiles = np.quantile(survival, np.linspace(0, 1, n_bins + 1)[1:-1])
    bins = np.digitize(survival, quantiles)

    print(f'OS CSV: {len(rows)} patients with survival data.')
    print(f'Survival days  min={survival.min():.0f}  median={np.median(survival):.0f}  max={survival.max():.0f}')
    print(f'Quantile thresholds ({n_bins} bins): {np.round(quantiles).astype(int).tolist()}')
    print(f'Class distribution: {np.bincount(bins, minlength=n_bins).tolist()}')
    return {pid: int(b) for pid, b in zip(pids, bins)}, quantiles


def group_by_patient(dataset: BraTS2020PNGDataset) -> dict[str, list[int]]:
    groups: dict[str, list[int]] = {}
    for i, (t2_path, _) in enumerate(dataset.samples):
        pid = Path(t2_path).parent.name
        groups.setdefault(pid, []).append(i)
    return groups


def build_patient_features(
    dataset: BraTS2020PNGDataset,
    groups: dict[str, list[int]],
    char_model,
    num_slots: int,
    device: torch.device,
    slot_model,
    agg_mode: str = 'sum',
) -> dict[str, np.ndarray]:
    feats: dict[str, np.ndarray] = {}
    with torch.no_grad():
        for pid, idxs in tqdm(groups.items(), desc=f'Characterising (trained, {agg_mode})'):
            acc = char_model.default_case().astype(np.int64)
            for i in idxs:
                t2, _seg = dataset[i]
                _, _, _, _, y_hat = slot_model(t2.unsqueeze(0).to(device))
                slots = y_hat[0].cpu()
                v = char_model.characterisation_transform(slots).astype(np.int64)
                if agg_mode == 'max':
                    acc = np.maximum(acc, v)
                else:
                    acc = acc + v
            feats[pid] = acc
    return feats


def _dedup(feats: np.ndarray, out: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    unique, inverse = np.unique(feats, axis=0, return_inverse=True)
    dedup_out = np.array([
        int(np.round(np.median(out[inverse == i])))
        for i in range(len(unique))
    ])
    return unique, dedup_out


def run_aacbr(
    train_feats: np.ndarray,
    train_out: np.ndarray,
    test_feats: np.ndarray,
    test_out: np.ndarray,
    char_model,
    cfg: BraTSOutcomeConfig,
    n_bins: int,
    *,
    strategy: str = 'ordinal',
    strict: bool = True,
    use_supports: bool = False,
    smart_default: bool = False,
    dedup: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    ls_fn = lambda a, b: char_model.less_specific(a, b, strict=strict)

    def make_model(k):
        default_out = (1 if k == 0 else 0) if smart_default else cfg.default_outcome
        return AACBRParallel(
            less_specific=ls_fn,
            default_case=char_model.default_case(),
            default_outcome=default_out,
            include_supports=use_supports,
            supported_attack_chain=use_supports,
        )

    models = {k: make_model(k) for k in range(n_bins)}

    if strategy == 'ordinal':
        f, o = _dedup(train_feats, train_out) if dedup else (train_feats, train_out)
        for k in range(n_bins):
            models[k].fit(f, (o >= k).astype(int))
        binary = np.stack([models[k].predict(test_feats) for k in range(n_bins)], axis=1)
        preds = binary[:, 1:].sum(axis=1)

    elif strategy == 'flat':
        f, o = _dedup(train_feats, train_out) if dedup else (train_feats, train_out)
        for k in range(n_bins):
            models[k].fit(f, (o == k).astype(int))
        binary = np.stack([models[k].predict(test_feats) for k in range(n_bins)], axis=1)
        preds = np.array([
            max(np.flatnonzero(binary[i] == 1), default=cfg.default_class)
            for i in range(len(test_feats))
        ])

    elif strategy == 'tournament_flat':
        for k in range(n_bins - 1):
            mask = train_out >= k
            feats_k = train_feats[mask]
            out_k = (train_out[mask] == k).astype(int)
            if dedup:
                feats_k, out_k = _dedup(feats_k, out_k)
            models[k].fit(feats_k, out_k)
        preds = np.full(len(test_feats), n_bins - 1, dtype=int)
        unclassified = np.ones(len(test_feats), dtype=bool)
        for k in range(n_bins - 1):
            preds_k = models[k].predict(test_feats[unclassified])
            newly = np.zeros(len(test_feats), dtype=bool)
            newly[unclassified] = preds_k == 1
            preds[newly] = k
            unclassified[newly] = False

    elif strategy == 'tournament_tree':
        for k in range(n_bins - 1):
            mask = train_out >= k
            feats_k = train_feats[mask]
            out_k = (train_out[mask] > k).astype(int)
            if dedup:
                feats_k, out_k = _dedup(feats_k, out_k)
            models[k].fit(feats_k, out_k)
        preds = np.full(len(test_feats), n_bins - 1, dtype=int)
        at_node = np.ones(len(test_feats), dtype=bool)
        for k in range(n_bins - 1):
            preds_k = models[k].predict(test_feats[at_node])
            left_mask = np.zeros(len(test_feats), dtype=bool)
            left_mask[at_node] = preds_k == 0
            preds[left_mask] = k
            at_node[left_mask] = False

    else:
        raise ValueError(f'Unknown strategy: {strategy!r}')

    return test_out, preds


def report(tag: str, y_true: np.ndarray, y_pred: np.ndarray, n_bins: int) -> dict:
    from sklearn.metrics import (
        accuracy_score, balanced_accuracy_score,
        confusion_matrix, classification_report,
    )
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mae     = np.abs(y_true.astype(float) - y_pred.astype(float)).mean()
    labels  = list(range(n_bins))
    cm      = confusion_matrix(y_true, y_pred, labels=labels)
    report_dict = classification_report(y_true, y_pred, labels=labels, zero_division=0, digits=3, output_dict=True)
    macro = report_dict.get('macro avg', {})
    print(f'\n=== {tag} ===')
    print(f'Accuracy: {acc:.4f}   Balanced Acc: {bal_acc:.4f}   Ordinal MAE: {mae:.4f}')
    print('Confusion matrix (rows=true, cols=pred):')
    print(cm)
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0, digits=3))
    return {
        'accuracy':          float(acc),
        'balanced_accuracy': float(bal_acc),
        'f1':                float(macro.get('f1-score', 0.0)),
        'precision':         float(macro.get('precision', 0.0)),
        'recall':            float(macro.get('recall', 0.0)),
        'ordinal_mae':       float(mae),
        'confusion_matrix':  cm.tolist(),
        'per_class': {
            k: {m: v for m, v in vs.items() if m != 'support'}
            for k, vs in report_dict.items()
            if isinstance(vs, dict)
        },
    }

In [ ]:
# ── User configuration ─────────────────────────────────────────────────────────
CHECKPOINT    = '/path/to/BrainWear_Kareem/FYP/slot_attention/training_2d/models/checkpoints/brats_png_v22b_cen_match/ckpt.pt'   # <-- set to your .pt file
SEED          = 0
N_FOLDS       = 5
NUM_SLOTS     = 5
AGG_MODES     = ['sum', 'max']
STRATEGIES    = ['ordinal', 'flat', 'tournament_flat', 'tournament_tree']
USE_SUPPORTS  = [True, False]
SMART_DEFAULTS = [True, False]
# ───────────────────────────────────────────────────────────────────────────────

np.random.seed(SEED)
torch.manual_seed(SEED)

device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg        = BraTSOutcomeConfig.from_json(CONFIG)
slot_model = load_slot_model(CHECKPOINT, device)
dataset    = BraTS2020PNGDataset(data_dir=DATA_DIR, is_train=False)
all_groups = group_by_patient(dataset)
print(f'Device: {device} | Patients: {len(all_groups)} | Slices: {len(dataset)}')

# -------------------------------------------------------------------
# Phase 1: Pre-build features once per (char_model, n_bins, agg_mode)
# -------------------------------------------------------------------
print('Pre-building features for all model/bin/agg combinations...')
feature_cache: dict[tuple, tuple] = {}

for char_name, n_bins, agg_mode in iproduct(
    ['small', 'large', 'small_v2', 'large_v2'], [2, 3, 4, 5], AGG_MODES
):
    char_model = CHAR_MODELS[char_name]
    print(f'\n--- char={char_name} ({char_model.DIMS}D), n_bins={n_bins}, agg={agg_mode} ---')
    patient_to_class, _ = load_os_csv(OS_CSV, n_bins=n_bins)

    matched_pids   = sorted(pid for pid in all_groups if pid in patient_to_class)
    matched_groups = {pid: all_groups[pid] for pid in matched_pids}

    feats    = build_patient_features(dataset, matched_groups, char_model, NUM_SLOTS, device, slot_model, agg_mode)
    features = np.array([feats[p] for p in matched_pids])
    outcomes = np.array([patient_to_class[p] for p in matched_pids])

    unique_count = len(np.unique(features, axis=0))
    print(f'  {len(matched_pids)} patients | Unique feature vectors: {unique_count} / {len(features)}')

    feature_cache[(char_name, n_bins, agg_mode)] = (features, outcomes)

# -------------------------------------------------------------------
# Phase 2: 5-fold CV sweep over all combos
# -------------------------------------------------------------------
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, precision_recall_fscore_support

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

n_combos = len(feature_cache) * len(STRATEGIES) * len(USE_SUPPORTS) * len(SMART_DEFAULTS) * 2  # strict True/False
print(f'\n\nRunning {N_FOLDS}-fold CV sweep ({n_combos} configurations)...')
sweep_results = []

for (char_name, n_bins, agg_mode), strategy, strict, use_supports, smart_default in tqdm(
    list(iproduct(feature_cache.keys(), STRATEGIES, [True, False], USE_SUPPORTS, SMART_DEFAULTS)),
    desc='CV sweep',
):
    features, outcomes = feature_cache[(char_name, n_bins, agg_mode)]
    char_model = CHAR_MODELS[char_name]

    fold_accs, fold_maes, fold_bal_accs, fold_f1s, fold_precisions, fold_recalls = [], [], [], [], [], []
    for train_idx, test_idx in kf.split(features, outcomes):
        y_true, y_pred = run_aacbr(
            features[train_idx], outcomes[train_idx],
            features[test_idx],  outcomes[test_idx],
            char_model, cfg, n_bins,
            strategy=strategy, strict=strict,
            use_supports=use_supports, smart_default=smart_default,
            dedup=True,
        )
        fold_accs.append(float(accuracy_score(y_true, y_pred)))
        fold_maes.append(float(np.abs(y_true.astype(float) - y_pred.astype(float)).mean()))
        fold_bal_accs.append(float(balanced_accuracy_score(y_true, y_pred)))
        p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
        fold_f1s.append(float(f))
        fold_precisions.append(float(p))
        fold_recalls.append(float(r))

    sweep_results.append(dict(
        char=char_name, n_bins=n_bins, agg=agg_mode, strategy=strategy,
        strict=strict, use_supports=use_supports, smart_default=smart_default,
        mean_acc=np.mean(fold_accs), std_acc=np.std(fold_accs),
        mean_f1=float(np.mean(fold_f1s)), std_f1=float(np.std(fold_f1s)),
        mean_mae=np.mean(fold_maes), std_mae=np.std(fold_maes),
        fold_accs=fold_accs, fold_maes=fold_maes,
        fold_bal_accs=fold_bal_accs, fold_f1s=fold_f1s,
        fold_precisions=fold_precisions, fold_recalls=fold_recalls,
    ))

sweep_results.sort(key=lambda r: (-r['mean_f1'], r['mean_mae']))

# -------------------------------------------------------------------
# Phase 3: Summary table (all configurations, sorted by mean CV macro-F1)
# -------------------------------------------------------------------
print(f"\n{'char':<12} {'agg':<5} {'bins':>4} {'strategy':<16} {'strict':>6} {'sup':>5} {'smdef':>5}   {'F1 (mean±std)':>16}   {'Acc (mean±std)':>16}")
print('-' * 102)
for r in sweep_results:
    f1_str  = f"{r['mean_f1']:.4f}±{r['std_f1']:.4f}"
    acc_str = f"{r['mean_acc']:.4f}±{r['std_acc']:.4f}"
    print(
        f"{r['char']:<12} {r['agg']:<5} {r['n_bins']:>4} {r['strategy']:<16} "
        f"{str(r['strict']):>6} {str(r['use_supports']):>5} {str(r['smart_default']):>5}   "
        f"{f1_str:>16}   {acc_str:>16}"
    )

In [ ]:
# Best config per strategy -- read from sweep CV results (no recomputation)
print(f'Best {N_FOLDS}-fold CV config per strategy\n')
print(f"{'strategy':<16} {'char':<12} {'agg':<5} {'bins':>4} {'strict':>6} {'sup':>5} {'smdef':>5} {'unique':>9}   {'F1':>6} +/- {'std':>6}    {'Acc':>6} +/- {'std':>6}")
print('-' * 110)

for strategy in STRATEGIES:
    b = next(r for r in sweep_results if r['strategy'] == strategy)
    features_b, outcomes_b = feature_cache[(b['char'], b['n_bins'], b['agg'])]
    n_unique = len(np.unique(features_b, axis=0))
    print(
        f"{strategy:<16} {b['char']:<12} {b['agg']:<5} {b['n_bins']:>4} {str(b['strict']):>6} "
        f"{str(b['use_supports']):>5} {str(b['smart_default']):>5} "
        f"{f'{n_unique}/{len(features_b)}':>9}   "
        f"{b['mean_f1']:>6.4f} +/- {b['std_f1']:>6.3f}    "
        f"{b['mean_acc']:>6.4f} +/- {b['std_acc']:>6.3f}   "
    )

In [ ]:
# ── log to trained-model leaderboard (set SAVE_RESULTS = True to persist) ─────
SAVE_RESULTS = True
NOTES = ''

if SAVE_RESULTS:
    from aacbr.results_logger import log_result

    N_BINS_RANGE = sorted({r['n_bins'] for r in sweep_results})

    for n_bins in N_BINS_RANGE:
        for strategy in STRATEGIES:
            b = next(
                (r for r in sweep_results if r['strategy'] == strategy and r['n_bins'] == n_bins),
                None,
            )
            if b is None:
                continue

            features_b, outcomes_b = feature_cache[(b['char'], b['n_bins'], b['agg'])]
            dist = np.bincount(outcomes_b, minlength=b['n_bins'])
            if dist.min() == 0:
                print(f'Skipping {strategy} n_bins={n_bins}: degenerate binning {dist.tolist()} -- not logged.')
                continue

            char_model_b = CHAR_MODELS[b['char']]

            # Re-run CV collecting OOF predictions for confusion matrix / per-class metrics
            oof_true, oof_pred = [], []
            for train_idx, test_idx in kf.split(features_b, outcomes_b):
                yt, yp = run_aacbr(
                    features_b[train_idx], outcomes_b[train_idx],
                    features_b[test_idx],  outcomes_b[test_idx],
                    char_model_b, cfg, b['n_bins'],
                    strategy=b['strategy'], strict=b['strict'],
                    use_supports=b['use_supports'], smart_default=b['smart_default'],
                    dedup=True,
                )
                oof_true.extend(yt)
                oof_pred.extend(yp)

            oof_true = np.array(oof_true)
            oof_pred = np.array(oof_pred)

            oof_metrics = report(
                f"strategy={strategy} n_bins={n_bins} [char={b['char']}, agg={b['agg']}, strict={b['strict']}]",
                oof_true, oof_pred, b['n_bins'],
            )

            log_result(
                {
                    'eval_script': 'eval_trained_2d',
                    'config': {
                        'checkpoint': CHECKPOINT,
                        'char_model': b['char'],
                        'n_bins': b['n_bins'],
                        'agg_mode': b['agg'],
                        'num_slots': NUM_SLOTS,
                        'strategy': b['strategy'],
                        'strict': b['strict'],
                        'use_supports': b['use_supports'],
                        'smart_default': b['smart_default'],
                        'n_folds': N_FOLDS,
                        'seed': SEED,
                    },
                    'data_stats': {
                        'n_patients': int(len(outcomes_b)),
                        'class_distribution': dist.tolist(),
                    },
                    'metrics': {
                        'cv_mean_accuracy':          b['mean_acc'],
                        'cv_std_accuracy':           b['std_acc'],
                        'cv_mean_balanced_accuracy': float(np.mean(b['fold_bal_accs'])),
                        'cv_std_balanced_accuracy':  float(np.std(b['fold_bal_accs'])),
                        'cv_mean_f1':                float(np.mean(b['fold_f1s'])),
                        'cv_std_f1':                 float(np.std(b['fold_f1s'])),
                        'cv_mean_precision':         float(np.mean(b['fold_precisions'])),
                        'cv_std_precision':          float(np.std(b['fold_precisions'])),
                        'cv_mean_recall':            float(np.mean(b['fold_recalls'])),
                        'cv_std_recall':             float(np.std(b['fold_recalls'])),
                        'cv_mean_mae':               b['mean_mae'],
                        'cv_std_mae':                b['std_mae'],
                        'fold_accs':                 b['fold_accs'],
                        'fold_maes':                 b['fold_maes'],
                        'fold_balanced_accs':        b['fold_bal_accs'],
                        'fold_f1s':                  b['fold_f1s'],
                        'oof_accuracy':              oof_metrics['accuracy'],
                        'oof_balanced_accuracy':     oof_metrics['balanced_accuracy'],
                        'oof_f1':                    oof_metrics['f1'],
                        'oof_precision':             oof_metrics['precision'],
                        'oof_recall':                oof_metrics['recall'],
                        'oof_ordinal_mae':           oof_metrics['ordinal_mae'],
                        'confusion_matrix':          oof_metrics['confusion_matrix'],
                        'per_class':                 oof_metrics['per_class'],
                    },
                    'notes': NOTES,
                },
                leaderboard_path=DEFAULT_TRAINED_LEADERBOARD,
            )